<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 5.7: Hazards
**Prev: [A Five-Stage Pipeline](5.6_pipeline.ipynb)**<br>
**Next: [A Bus and a UART](5.8_bus_and_mmio.ipynb)**<br>

## Motivation
The pipeline of [Module 5.6](5.6_pipeline.ipynb) needs two nops between an instruction and the
next one that uses its result. That costs cycles, and it is not something to ask of a compiler
or of the official tests. This notebook makes the pipeline handle its hazards in hardware:

- it **forwards** a result to the instruction that needs it as soon as the result exists, instead
  of waiting until it is written to the register file;
- it **stalls** for one cycle when even forwarding is too late (a value loaded from memory, used
  right away), and while the memory has not answered;
- and it keeps flushing the wrong path after a jump or taken branch, as in 5.6.

The result runs any RV32I program as it is: the official tests of [Module 5.5](5.5_compliance.ipynb)
pass on it.

## Setup
If a cell fails or hangs, see [Troubleshooting](docs/troubleshooting.md).

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
interp.load.module(os.pwd / "source" / "load-riscv.sc")

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.simulator.PeekPokeAPI._
import riscv._

---
# Every hazard, and what resolves it

| Hazard | Example | Resolved by | Signal in `ForwardingCore` |
|---|---|---|---|
| A result is needed one instruction later | `li a0, 5` then `addi a1, a0, 1` | forwarding from EX/MEM to EX | `fromExMem` into `Forward` |
| A result is needed two instructions later | `li a0, 5`, any, `addi a1, a0, 1` | forwarding from MEM/WB to EX | `fromMemWb` into `Forward` |
| A result is needed three instructions later | the same, two between | a bypass in ID, as if the register file were written before it is read | `wbRd`, `wbValue` (5.6) |
| A loaded value is needed right away | `lw a0, 0(t0)` then `addi a1, a0, 1` | one-cycle stall: IF and ID hold, EX gets a bubble; then MEM/WB forwards | `loadUse` |
| A jump or taken branch | `beq` resolved in EX | predict not taken; flush IF/ID and ID/EX, fetch the target | `redirect`, `PipelineFlush` (5.6) |
| An instruction halts the core | `ecall` in MEM | flush IF/ID, ID/EX and EX/MEM; fetch no more | `haltInMem`, `PipelineFlush` (5.6) |
| The data memory answers late | a slow bus | everything holds (WB gets bubbles); EX keeps its forwarded operands | `memWait` |

The instruction memory must still answer in the same cycle: a slow fetch needs a core that keeps
its fetch request while a jump redirects it, which [Module 5.8](5.8_bus_and_mmio.ipynb) discusses with the bus. One more
detail: a store that needs a just-loaded value only as the data to store also waits a cycle,
although that value could be forwarded from MEM to MEM; the simpler rule costs a cycle, not
correctness.

Forwarding takes the *newest* result: when the instructions in both MEM and WB write the
register, the one in MEM is younger, and its value is the one the program means. And it never
forwards for `x0`, which stays 0 whatever an instruction writes to it.

---
# Exercise: forwarding

`riscv.Forward` chooses the value of one source register for the instruction in EX: from the
instruction one ahead (EX/MEM), from the one two ahead (MEM/WB), or the value read in ID.
Complete `riscv/src/main/scala/riscv/Forward.scala`, restart the kernel and run the notebook again
(the cells that build the core stop with a `NotImplementedError` until then, and also until the
exercises of 5.2-5.4 and 5.6 are done: the core uses them all).

In [ ]:
simulate(new Forward) { c =>
  def check(rs: Int, ex: Option[(Int, Int)], wb: Option[(Int, Int)], expected: Int) = {
    c.io.rs.poke(rs.U)
    c.io.regValue.poke(1.U)
    for ((port, r) <- Seq(c.io.exMem -> ex, c.io.memWb -> wb)) {
      port.valid.poke(r.isDefined.B)
      port.bits.rd.poke(r.map(_._1).getOrElse(0).U)
      port.bits.value.poke(r.map(_._2).getOrElse(0).U)
    }
    c.io.value.expect(expected.U, s"rs $rs, EX/MEM $ex, MEM/WB $wb")
  }
  check(5, None, None, 1) // nothing to forward: the register file's value
  check(5, Some(5 -> 2), None, 2)
  check(5, None, Some(5 -> 3), 3)
  check(5, Some(5 -> 2), Some(5 -> 3), 2) // the newest
  check(5, Some(6 -> 2), Some(5 -> 3), 3)
  check(5, Some(6 -> 2), Some(7 -> 3), 1) // neither is for rs
  check(0, Some(0 -> 2), Some(0 -> 3), 1) // never x0
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  def has(r: Valid[Result]) = r.valid &amp;&amp; r.bits.rd =/= 0.U &amp;&amp; r.bits.rd === io.rs
  io.value := MuxCase(io.regValue, Seq(has(io.exMem) -&gt; io.exMem.bits.value, has(io.memWb) -&gt; io.memWb.bits.value))
</pre></article></div></section></div>

---
# The core

`riscv.ForwardingCore` is the pipeline of 5.6 with these changes: EX takes its operands through two
`Forward`s; `loadUse` holds IF/ID and the pc and sends a bubble into EX; `memWait` holds every
stage, sends bubbles into WB, and keeps EX's forwarded operands; and a redirect waits while the
pipeline holds (a flush would remove the instruction that is waiting in EX).

In [ ]:
println(os.read(os.pwd / "riscv" / "src" / "main" / "scala" / "riscv" / "ForwardingCore.scala"))

---
# Testing the hazards

One short program per hazard of the table, without nops, each compared instruction by instruction
with the golden model. They include a store on the wrong path of a jump and a store behind a halt,
which must have no effect.

In [ ]:
val hazards = Seq(
  "EX/MEM to EX"             -> "li a0, 5\naddi a1, a0, 1\necall",
  "MEM/WB to EX"             -> "li a0, 5\nnop\naddi a1, a0, 1\necall",
  "the newest result wins"   -> "li a0, 1\nli a0, 2\nadd a1, a0, a0\necall",
  "x0 is not forwarded"      -> "addi zero, zero, 7\nadd a0, zero, zero\necall",
  "load-use"                 -> "la t0, d\nlw a0, 0(t0)\naddi a1, a0, 1\necall\n.data\nd: .word 41",
  "load then store"          -> "la t0, d\nlw a0, 0(t0)\nsw a0, 4(t0)\nlw a1, 4(t0)\necall\n.data\nd: .word 41, 0",
  "branch on a loaded value" -> "la t0, d\nlw a0, 0(t0)\nbnez a0, 1f\nebreak\n1: ecall\n.data\nd: .word 1",
  "jal link used at once"    -> "jal ra, 1f\nebreak\n1: addi a0, ra, 0\necall",
  "wrong-path stores"        -> "la t0, d\nj 1f\nsw t0, 0(t0)\n1: lw a0, 0(t0)\necall\n.data\nd: .word 7",
  "a store behind a halt"    -> "la t0, d\nebreak\nsw t0, 0(t0)\n.data\nd: .word 7"
)
simulate(new ForwardingSystem) { c =>
  for ((name, source) <- hazards) {
    CoSim.compare(c, Asm(source)).foreach(d => sys.error(s"$name: $d"))
    println(s"ok: $name")
  }
}
println("SUCCESS!!")

---
# The official tests, on the pipeline

The rv32ui tests, as they are (no nops), on the pipelined core, again compared instruction by
instruction with the golden model:

In [ ]:
val tests = Programs.images("programs/riscv-tests/rv32ui") // each test, by name, assembled
val start = System.nanoTime
simulate(new ForwardingSystem) { c =>
  val failed = tests.filter { case (name, image) =>
    val bad = CoSim.compare(c, image).isDefined || CoSim.reg(c, riscv.Reg("a0")) != 0
    if (bad) println(s"FAIL: $name")
    bad
  }
  println(
    f"${tests.size - failed.size} of ${tests.size} pass on the pipeline, in ${(System.nanoTime - start) / 1e9}%.0f s"
  )
  assert(failed.isEmpty)
}
println("SUCCESS!!")

---
# How much faster?

A **cycle counter** (`CoSim.timed`) gives the cycles a program takes; divided by the instructions
it retires, that is its CPI, cycles per instruction. The single-cycle core takes exactly one
cycle per instruction, but a long clock period; the pipeline has a short one, and a CPI above 1:
it loses four cycles filling the pipeline, one per load-use stall, and two for the instructions
flushed after each jump or taken branch. Without forwarding (5.6), the nops are instructions too,
and the CPI per useful instruction is far worse:

In [ ]:
val programs = Seq("fib", "fact").map(n => n -> Programs.source("programs/src", n))
// the cycles each program takes on one core (one simulation for all the programs)
def measure(system: => RiscvSystem, images: Seq[Seq[Long]]): Seq[Int] = {
  var cycles = Seq.empty[Int]
  simulate(system) { c =>
    cycles = images.map { image =>
      CoSim.load(c, image)
      val (trace, n) = CoSim.timed(c)
      assert(trace.lastOption.exists(_.halt.isDefined), "the program did not halt")
      n
    }
  }
  cycles
}
val padded  = measure(new PipelinedSystem, programs.map(p => Asm.padded(p._2, 2)))
val forward = measure(new ForwardingSystem, programs.map(p => Asm(p._2)))
for ((((name, source), p), f) <- programs.zip(padded).zip(forward)) {
  val useful = new IsaSim(Asm(source)).run().size // also the single-cycle core's cycles
  println(
    f"$name%-5s $useful%4d instructions: single-cycle $useful%4d cycles, " +
      f"5.6 with nops $p%4d (CPI ${p.toDouble / useful}%.2f), 5.7 $f%4d (CPI ${f.toDouble / useful}%.2f)"
  )
}

The pipeline needs more cycles than the single-cycle core, but each of its cycles is shorter: one
stage instead of the whole path, so ideally up to five times shorter, and less in practice (the
stages are not equally long, the stage registers add delay, and forwarding lengthens EX). Whether
it runs a program in less time depends on those delays, which a synthesis tool reports (Module
5.10).

---
# You're done!

[Return to the top.](#top)